# Inference Pipeline

YOLO-based object counting over folders of video-still images.

## Libraries

In [ ]:
import os
import datetime
import pickle
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

from PIL import Image, ExifTags
import pandas as pd
from ultralytics import YOLO

import logging

class SuppressDeprecationFilter(logging.Filter):
    def filter(self, record):
        return "'half' is deprecated" not in record.getMessage()

logging.getLogger("ultralytics").addFilter(SuppressDeprecationFilter())

# Configs

## 1. Configuration Layer

In [ ]:
SOURCE_FOLDERS = [
    ("/content/local_video_stills", "results1"),
    ("/content/local_video_stills2", "results2"),
    ("/content/local_video_stills3", "results3"),
]

MODEL_WEIGHTS = "yolo26m.pt"  # Change to your custom weights path later
TARGET_CATEGORIES = ["person", "bicycle", "motorcycle", "car", "traffic light"]

USE_HALF = True
CONF_THRESHOLD = 0.15
IMAGE_LIMIT = 1642  # cap per run for testing; set to None for no cap

# Where human-readable CSVs go (one per folder)
OUTPUT_FOLDER = "/content/drive/MyDrive/datasets/"
# Where machine-readable copies of each run's dataframe go, for a downstream
# script to load directly (pd.read_pickle) without re-parsing a CSV.
PERSIST_FOLDER = os.path.join(OUTPUT_FOLDER, "pickled_runs")
# Central experiment log, one row appended per run.
CENTRAL_LOG_PATH = os.path.join(OUTPUT_FOLDER, "cv_experiment_log.csv")

## 2. Metadata Extraction

In [ ]:
def get_image_timestamp(file_path):
    """
    Attempts to get the timestamp from EXIF metadata.
    Falls back to OS modification time if EXIF is unavailable.
    """
    try:
        img = Image.open(file_path)
        exif_data = img._getexif()
        if exif_data:
            # 36867 is the EXIF tag for DateTimeOriginal
            for tag, value in exif_data.items():
                if tag == 36867:
                    return value
    except Exception:
        pass  # Handle or log image opening errors if necessary

    # Fallback: OS modification time
    mtime = os.path.getmtime(file_path)
    return datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')

# Pipeline Def

## 3. Vision Pipeline Function

This function's only job is to run the model over a folder of images and hand back structured data. It doesn't know or care where that data ends up (CSV, pickle, a logging spreadsheet, etc.) — that's the concern of the Reporting/Persistence/Logging section below.

In [ ]:
@dataclass
class PipelineRun:
    """Everything one call to run_vision_pipeline produced, bundled together
    so it can be handed as a single object to whatever consumes it next
    (CSV writer, pickle writer, experiment logger, ...)."""
    df: pd.DataFrame
    source_folder: str
    model_weights: str
    half_precision: bool
    conf_threshold: float
    target_categories: List[str]
    images_processed: int
    avg_inference_ms: float
    run_timestamp: str = field(
        default_factory=lambda: datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    )


def run_vision_pipeline(
    source_folder,
    weights,
    target_cats,
    conf=CONF_THRESHOLD,
    half=USE_HALF,
    limit=IMAGE_LIMIT,
) -> PipelineRun:
    print(f"Loading model: {weights}...")
    model = YOLO(weights)

    model_classes = model.names
    pipeline_results = []
    valid_extensions = {".jpg", ".jpeg", ".png"}
    folder_path = Path(source_folder)

    processed_count = 0
    total_inference_time = 0  # Track cumulative engine latency

    # Sort files alphabetically for a stable, reproducible order.
    # For chronological order instead, use: sorted(folder_path.iterdir(), key=os.path.getmtime)
    for img_path in sorted(folder_path.iterdir()):
        if img_path.suffix.lower() not in valid_extensions:
            continue

        if limit is not None and processed_count >= limit:
            print(f"Reached image limit of {limit}. Stopping.")
            break

        filename = img_path.name
        timestamp = get_image_timestamp(img_path)
        cat_counts = {cat: 0 for cat in target_cats}

        results = model(img_path, device="cuda", half=half, conf=conf, verbose=False)[0]

        # results.speed is a dict of stage->milliseconds: preprocess, inference, postprocess
        total_inference_time += results.speed['inference']

        # Aggregation: count only the categories we care about
        for box in results.boxes:
            cls_id = int(box.cls[0])
            cls_name = model_classes[cls_id]
            if cls_name in cat_counts:
                cat_counts[cls_name] += 1

        row = [filename] + [cat_counts[cat] for cat in target_cats] + [timestamp]
        pipeline_results.append(row)

        processed_count += 1

    avg_time = (total_inference_time / processed_count) if processed_count > 0 else 0.0

    column_headers = ["filename"] + target_cats + ["timestamp"]
    df = pd.DataFrame(pipeline_results, columns=column_headers)

    return PipelineRun(
        df=df,
        source_folder=str(source_folder),
        model_weights=weights,
        half_precision=half,
        conf_threshold=conf,
        target_categories=list(target_cats),
        images_processed=processed_count,
        avg_inference_ms=avg_time,
    )

# Reporting, Persistence & Logging

## 4. Reporting / Persistence / Logging

Three separate, single-purpose functions instead of one do-everything logger. The vision pipeline above never touches disk except to read images — everything below decides what happens to its output.

In [ ]:
def print_run_report(run: PipelineRun):
    """Console summary for a single run. Purely informational."""
    print("\n==========================================")
    print("PERFORMANCE STATS:")
    print(f"Source folder: {run.source_folder}")
    print(f"Images Processed: {run.images_processed}")
    print(f"Average Inference Time: {run.avg_inference_ms:.2f} ms per image")
    print("==========================================\n")


def save_run_csv(run: PipelineRun, output_folder, filename) -> Path:
    """Human-readable export for this run's dataframe."""
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename
    run.df.to_csv(output_path, index=False)
    print(f"Saved CSV to {output_path}")
    return output_path


def persist_run_dataframe(run: PipelineRun, persist_folder, filename) -> Path:
    """
    Machine-readable export for a downstream script. Pickling (instead of
    CSV) preserves dtypes/index and can be loaded with a single call:
        df = pd.read_pickle(path)
    with no re-parsing or dtype-guessing needed on the reading side.
    """
    Path(persist_folder).mkdir(parents=True, exist_ok=True)
    persist_path = Path(persist_folder) / filename
    run.df.to_pickle(persist_path)
    print(f"Persisted dataframe to {persist_path}")
    return persist_path


def log_experiment(run: PipelineRun, log_path):
    """
    Appends one row summarizing this run's parameters + aggregate detection
    counts to a central CSV, so multiple runs can be compared over time.
    """
    run_summary = {
        "timestamp": run.run_timestamp,
        "source_folder": run.source_folder,
        "model_weights": run.model_weights,
        "half_precision": run.half_precision,
        "confidence_threshold": run.conf_threshold,
        "images_processed": run.images_processed,
        "avg_inference_ms": round(run.avg_inference_ms, 2),
    }
    for cat in run.target_categories:
        run_summary[f"total_{cat}_detected"] = int(run.df[cat].sum())

    new_run_df = pd.DataFrame([run_summary])

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    if log_path.exists():
        existing_df = pd.read_csv(log_path)
        updated_df = pd.concat([existing_df, new_run_df], ignore_index=True)
        updated_df.to_csv(log_path, index=False)
    else:
        new_run_df.to_csv(log_path, index=False)

    print(f"Experiment metrics logged to: {log_path}")
    print(new_run_df.to_string(index=False))

# Main

## 5. Orchestration

In [ ]:
def process_folder(source_folder, csv_stem):
    """Run the CV pipeline on one folder and fan the result out to every
    downstream concern (report, CSV, pickle, central log)."""
    Path(source_folder).mkdir(parents=True, exist_ok=True)

    run = run_vision_pipeline(
        source_folder,
        MODEL_WEIGHTS,
        TARGET_CATEGORIES,
        conf=CONF_THRESHOLD,
        half=USE_HALF,
        limit=IMAGE_LIMIT,
    )

    print("\n--- Pipeline Execution Complete ---")
    print(run.df.head())

    print_run_report(run)
    save_run_csv(run, OUTPUT_FOLDER, f"{csv_stem}.csv")
    persist_run_dataframe(run, PERSIST_FOLDER, f"{csv_stem}.pkl")
    log_experiment(run, CENTRAL_LOG_PATH)

    return run

In [ ]:
all_runs = [process_folder(folder, stem) for folder, stem in SOURCE_FOLDERS]

# From here, any run's numerical data can be converted to a
# PyTorch/NumPy tensor, e.g.:
# tensor_data = all_runs[0].df[TARGET_CATEGORIES].to_numpy()